# 03. Modeling: Klasterisasi & Evaluasi Algoritma
## Penentuan k Optimal, K-Means, GMM, Agglomerative & Proyeksi PCA

Notebook ini memuat tahapan **Modeling & Evaluation**:
1. Penentuan jumlah $k$ optimal menggunakan **Elbow Method (Inertia)** dan **Silhouette Analysis**.
2. Melatih dan membandingkan multi-algoritma: **K-Means**, **Gaussian Mixture Model (GMM)**, dan **Agglomerative Hierarchical Clustering**.
3. Evaluasi metrik kualitas: **Silhouette Score**, **Davies-Bouldin Index**, dan **Calinski-Harabasz**.
4. Reduksi dimensi **PCA 2D dan 3D** untuk memvisualisasikan separasi klaster.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import BPSDataPreprocessor
from src.features import compute_bps_regional_features
from src.clustering import BPSClusterEngine
from src.evaluation import evaluate_k_range, benchmark_clustering_algorithms

# Pipeline Persiapan Data
df_raw = pd.read_csv('../data/raw/real_bps_kabupaten_kota.csv')
p = BPSDataPreprocessor()
df_clean = p.clean_and_impute(df_raw)
df_clean = p.handle_outliers_iqr(df_clean)
df_feat = compute_bps_regional_features(df_clean)

clustering_cols = [
    'persentase_penduduk_miskin', 'garis_kemiskinan', 'indeks_kedalaman_p1', 'indeks_keparahan_p2',
    'pengeluaran_per_kapita', 'indeks_pembangunan_manusia_ipm', 'persentase_sanitasi_layak',
    'persentase_air_minum_layak', 'persentase_rumah_layak_huni', 'indeks_infrastruktur_dasar',
    'indeks_kemiskinan_multidimensi_ikm'
]
X_scaled, _ = p.scale_features(df_feat, clustering_cols)

In [ ]:
# 1. Evaluasi Rentang k (2 sampai 8)
k_eval_df = evaluate_k_range(X_scaled, k_min=2, k_max=8)

fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Jumlah Klaster (k)', fontweight='bold')
ax1.set_ylabel('Inertia (Elbow)', color=color, fontweight='bold')
ax1.plot(k_eval_df['k'], k_eval_df['inertia'], 'o-', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color, fontweight='bold')
ax2.plot(k_eval_df['k'], k_eval_df['silhouette_score'], 's--', color=color, linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Evaluasi Jumlah Klaster Optimal (Elbow vs Silhouette Score)', fontsize=13, fontweight='bold')
plt.show()
k_eval_df

In [ ]:
# 2. Benchmarking Multi-Algoritma (k=4)
benchmark_res = benchmark_clustering_algorithms(X_scaled, n_clusters=4)
benchmark_res

In [ ]:
# 3. Klasterisasi K-Means dan Proyeksi PCA 2D
engine = BPSClusterEngine(n_clusters=4, random_state=42)
labels = engine.fit_kmeans(X_scaled)
df_feat['cluster'] = labels

df_pca_2d, df_pca_3d = engine.compute_pca_projections(X_scaled)
df_feat['PCA1'] = df_pca_2d['PCA1']
df_feat['PCA2'] = df_pca_2d['PCA2']

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_feat, x='PCA1', y='PCA2', hue='cluster', palette='tab10', s=70, alpha=0.9
)
plt.title('Visualisasi 2D Proyeksi PCA Klaster Kesejahteraan BPS (k=4)', fontsize=13, fontweight='bold')
plt.show()